This notebook is for data preprocessing check.


Dropped Time

Removed features with more than 50% missing values

Imputed remaining missing values with median

Removed constant features

Performed stratified 70/15/15 split

Standardized features using training statistics

In [4]:
#import package
import pandas as pd
import numpy as np
import sys
import os
from sklearn.model_selection import train_test_split

sys.path.append(os.path.abspath(".."))
from src.utils import RANDOM_SEED, TRAIN_RATIO, VAL_RATIO, TEST_RATIO

In [5]:
print(sys.version)

3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]


In [6]:
df_raw = pd.read_csv("../data/uci-secom.csv")
df = df_raw.copy() #use copy to not change the origianl file

In [7]:
print("Shape:", df.shape)

Shape: (1567, 592)


In [8]:
#delete pass/fail for not data leak
# Remove Time from the input features.
# We use only sensor-related features for the first baseline experiment.
y = df["Pass/Fail"].copy()
X = df.drop(columns=["Pass/Fail"]).copy()

print("Before dropping Time:", X.shape)

X = X.drop(columns=["Time"]).copy()

print("After dropping Time:", X.shape)

Before dropping Time: (1567, 591)
After dropping Time: (1567, 590)


In [9]:
#drop the data that have missing ratio bigger than 50%
missing_ratio = X.isnull().mean()
cols_to_drop = missing_ratio[missing_ratio > 0.5].index
print("Number of columns to drop:", len(cols_to_drop))

Number of columns to drop: 28


In [10]:
X = X.drop(columns=cols_to_drop).copy()
print("Shape after dropping high-missing columns:", X.shape)

Shape after dropping high-missing columns: (1567, 562)


In [11]:
#check the missing value percentage for the data that remains
print("Total missing values in X:", X.isnull().sum().sum())
missing_ratio_after = X.isnull().mean().sort_values(ascending=False)
missing_ratio_after.head(10)

Total missing values in X: 11683


385    0.456286
247    0.456286
112    0.456286
519    0.456286
566    0.174218
567    0.174218
568    0.174218
569    0.174218
563    0.174218
562    0.174218
dtype: float64

In [12]:
#fill the data with median (for each column)
X = X.fillna(X.median())
print("Total missing values after median imputation:", X.isnull().sum().sum())

Total missing values after median imputation: 0


In [13]:
#find the col that have same numbers in it 
constant_cols = [col for col in X.columns if X[col].nunique() == 1]

print("Number of constant features:", len(constant_cols))
print("Constant features:", constant_cols[:10])

Number of constant features: 116
Constant features: ['5', '13', '42', '49', '52', '69', '97', '141', '149', '178']


In [14]:
X = X.drop(columns=constant_cols).copy()
print("Shape after dropping constant features:", X.shape)

Shape after dropping constant features: (1567, 446)


In [15]:
print(y.value_counts())
print(y.value_counts(normalize=True))

Pass/Fail
-1    1463
 1     104
Name: count, dtype: int64
Pass/Fail
-1    0.933631
 1    0.066369
Name: proportion, dtype: float64


In [16]:
print (RANDOM_SEED)

559


In [17]:
# First split: train vs temporary
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=(VAL_RATIO + TEST_RATIO),
    random_state=RANDOM_SEED,
    stratify=y
)

# Second split: validation vs test from the temporary set
temp_test_ratio = TEST_RATIO / (VAL_RATIO + TEST_RATIO)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=temp_test_ratio,
    random_state=RANDOM_SEED,
    stratify=y_temp
)

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("X_test shape:", X_test.shape)

X_train shape: (1096, 446)
X_val shape: (235, 446)
X_test shape: (236, 446)


In [18]:
print("Train label distribution:")
print(y_train.value_counts())
print(y_train.value_counts(normalize=True))

print("\nValidation label distribution:")
print(y_val.value_counts())
print(y_val.value_counts(normalize=True))

print("\nTest label distribution:")
print(y_test.value_counts())
print(y_test.value_counts(normalize=True))

Train label distribution:
Pass/Fail
-1    1023
 1      73
Name: count, dtype: int64
Pass/Fail
-1    0.933394
 1    0.066606
Name: proportion, dtype: float64

Validation label distribution:
Pass/Fail
-1    220
 1     15
Name: count, dtype: int64
Pass/Fail
-1    0.93617
 1    0.06383
Name: proportion, dtype: float64

Test label distribution:
Pass/Fail
-1    220
 1     16
Name: count, dtype: int64
Pass/Fail
-1    0.932203
 1    0.067797
Name: proportion, dtype: float64


In [19]:
# Compute mean and standard deviation from the training set only
train_mean = X_train.mean()
train_std = X_train.std()

# Avoid division by zero just in case
train_std = train_std.replace(0, 1)

# Standardize train, validation, and test sets
X_train_scaled = (X_train - train_mean) / train_std
X_val_scaled = (X_val - train_mean) / train_std
X_test_scaled = (X_test - train_mean) / train_std

print("X_train_scaled shape:", X_train_scaled.shape)
print("X_val_scaled shape:", X_val_scaled.shape)
print("X_test_scaled shape:", X_test_scaled.shape)

X_train_scaled shape: (1096, 446)
X_val_scaled shape: (235, 446)
X_test_scaled shape: (236, 446)


In [20]:
print("Mean of first 5 scaled training features:")
print(X_train_scaled.iloc[:, :5].mean())

print("\nStd of first 5 scaled training features:")
print(X_train_scaled.iloc[:, :5].std())

Mean of first 5 scaled training features:
0    3.591612e-15
1   -1.120758e-15
2    9.060879e-15
3   -1.166950e-16
4   -3.241527e-18
dtype: float64

Std of first 5 scaled training features:
0    1.0
1    1.0
2    1.0
3    1.0
4    1.0
dtype: float64


In [23]:
import pickle

preprocessed_data = {
    "X_train": X_train_scaled,
    "X_val": X_val_scaled,
    "X_test": X_test_scaled,
    "y_train": y_train,
    "y_val": y_val,
    "y_test": y_test
}

with open("../data/preprocessed_data.pkl", "wb") as f:
    pickle.dump(preprocessed_data, f)

print("Preprocessed data saved to ../data/preprocessed_data.pkl")

Preprocessed data saved to ../data/preprocessed_data.pkl
